In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from agent_lab.model_hub import LLM_FAST

model = init_chat_model(**LLM_FAST)

In [2]:
system_msg = SystemMessage("You are a helpful assistant.")
human_msg = HumanMessage("Hello, how are you?")

# Use with chat models
messages = [system_msg, human_msg]
response = model.invoke(messages)  # Returns AIMessage

In [3]:
response.pretty_print()

================================== Ai Message ==================================

I'm doing well, thank you! How can I assist you today?


#### AIMessage

当模型调用工具时，它们会被包含在 AIMessage 中：

In [4]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

model_with_tools = model.bind_tools([get_weather])

In [5]:
messages = [HumanMessage("What's the weather in Boston?")]
ai_msg = model_with_tools.invoke(messages)

In [6]:
ai_msg.content_blocks

[{'type': 'tool_call',
  'id': 'call_00_8HltUEAK8B4BG0iPVYQqASDg',
  'name': 'get_weather',
  'args': {'location': 'Boston'}}]

In [7]:
ai_msg.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Boston'},
  'id': 'call_00_8HltUEAK8B4BG0iPVYQqASDg',
  'type': 'tool_call'}]

In [8]:
ai_msg.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_8HltUEAK8B4BG0iPVYQqASDg)
 Call ID: call_00_8HltUEAK8B4BG0iPVYQqASDg
  Args:
    location: Boston


Token 使用情况

In [9]:
ai_msg.usage_metadata

{'input_tokens': 280,
 'output_tokens': 44,
 'total_tokens': 324,
 'input_token_details': {'cache_read': 0},
 'output_token_details': {}}

#### ToolMessage

In [10]:
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result

In [11]:
response.pretty_print()

================================== Ai Message ==================================

The weather in San Francisco is currently **sunny** with a temperature of **72°F** (about 22°C).
